In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q decord

import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import transforms
from transformers import TimesformerModel, TimesformerConfig
from decord import VideoReader, cpu

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 136.0 MB/s eta 0:00:00


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

class FightTimeSformer(nn.Module):
    def __init__(self, backbone, num_classes=2):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(
            backbone.config.hidden_size,
            num_classes
        )

    def forward(self, x):
        outputs = self.backbone(x)
        cls_token = outputs.last_hidden_state[:, 0]
        return self.classifier(cls_token)


config = TimesformerConfig.from_pretrained(
    "facebook/timesformer-base-finetuned-k400"
)
config.num_frames = 8
config.image_size = 224

backbone = TimesformerModel.from_pretrained(
    "facebook/timesformer-base-finetuned-k400",
    config=config
)

model = FightTimeSformer(backbone)
model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/securevision_fight_timesformer_hardneg.pth",
        map_location=device
    )
)

model = model.to(device)
model.eval()
model.backbone.eval()

print("✅ Model loaded successfully")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/486M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/486M [00:00<?, ?B/s]

✅ Model loaded successfully


In [ ]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [ ]:
import torch.nn.functional as F

def predict_video_multiclip_probs(video_path, clips=5):
    vr = VideoReader(video_path, ctx=cpu(0))
    total_frames = len(vr)

    violence_probs = []

    for i in range(clips):
        if total_frames > 8:
            start = int(i * (total_frames - 8) / clips)
            indices = np.arange(start, start + 8)
        else:
            indices = np.linspace(0, total_frames - 1, 8).astype(int)

        frames = vr.get_batch(indices).asnumpy()
        frames = [transform(f) for f in frames]

        video = torch.stack(frames)
        video = video.permute(1, 0, 2, 3)
        video = video.unsqueeze(0).to(device)
        video = video.permute(0, 2, 1, 3, 4)

        with torch.no_grad():
            logits = model(video)
            probs = F.softmax(logits, dim=1)
            violence_probs.append(probs[0, 1].item())  # class=1

    return violence_probs


In [ ]:
CONF_THRESH = 0.75     # violence confidence threshold
MIN_HITS    = 3        # minimum violent clips required
TOTAL_CLIPS = 5

def decide_violence(video_path):
    probs = predict_video_multiclip_probs(video_path, clips=TOTAL_CLIPS)

    violent_hits = sum(p >= CONF_THRESH for p in probs)

    if violent_hits >= MIN_HITS:
        return 1  # Violence
    else:
        return 0  # NonViolence


In [ ]:
TEST_ROOT = "/content/drive/MyDrive/Testing"

label_map = {
    "NonViolence": 0,
    "Violence": 1
}

video_paths = []
true_labels = []

for cls, label in label_map.items():
    cls_dir = os.path.join(TEST_ROOT, cls)

    for ext in ["*.mp4", "*.avi", "*.mov", "*.mkv", "*.mpeg", "*.mpg"]:
        for vp in glob.glob(os.path.join(cls_dir, ext)):
            video_paths.append(vp)
            true_labels.append(label)

print("Total test videos:", len(video_paths))


Total test videos: 311


In [ ]:
pred_labels = []

for vp in video_paths:
    try:
        pred = decide_violence(vp)
        pred_labels.append(pred)
    except Exception as e:
        print("Error:", vp)
        pred_labels.append(0)


In [ ]:
acc = accuracy_score(true_labels, pred_labels)
prec = precision_score(true_labels, pred_labels)
rec = recall_score(true_labels, pred_labels)
f1 = f1_score(true_labels, pred_labels)

print("========== Evaluation Results ==========")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-score  : {f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, pred_labels))

print("\nClassification Report:")
print(classification_report(
    true_labels,
    pred_labels,
    target_names=["NonViolence", "Violence"]
))


========== Evaluation Results ==========
Accuracy  : 0.6881
Precision : 0.6480
Recall    : 0.7733
F1-score  : 0.7052

Confusion Matrix:
[[ 98  63]
 [ 34 116]]

Classification Report:
              precision    recall  f1-score   support

 NonViolence       0.74      0.61      0.67       161
    Violence       0.65      0.77      0.71       150

    accuracy                           0.69       311
   macro avg       0.70      0.69      0.69       311
weighted avg       0.70      0.69      0.69       311

